In [1]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.naive_bayes import CategoricalNB, GaussianNB

In [4]:
#import dataframe
buys = pd.read_csv('buysComputer.csv')
buys.head(15)

,age,income,student,credit_rating,buys_computer
0,<=30,high,no,fair,no
1,<=30,high,no,excellent,no
2,31-40,high,no,fair,yes
3,>40,medium,no,fair,yes
4,>40,low,yes,fair,yes
5,>40,low,yes,excellent,no
6,31-40,low,yes,excellent,yes
7,<=30,medium,no,fair,no
8,<=30,low,yes,fair,yes
9,>40,medium,yes,fair,yes


In [5]:
[buys[col].value_counts() for col in buys.columns]

[age
 <=30     5
 >40      5
 31-40    4
 Name: count, dtype: int64,
 income
 medium    6
 high      4
 low       4
 Name: count, dtype: int64,
 student
 no     7
 yes    7
 Name: count, dtype: int64,
 credit_rating
 fair         8
 excellent    6
 Name: count, dtype: int64,
 buys_computer
 yes    9
 no     5
 Name: count, dtype: int64]

In [8]:
maps = {
    "age" : {"<=30": 0, "31-40": 1, ">40": 2},
    "income" : {"low": 0, "medium": 1, "high": 2},
    "student" : {"no": 0, "yes": 1},
    "credit_rating" : {"fair": 0, "excellent": 1}
}

x = buys[list(maps)].apply(lambda col: col.map(maps[col.name]))
y = buys['buys_computer']

x

,age,income,student,credit_rating
0,0,2,0,0
1,0,2,0,1
2,1,2,0,0
3,2,1,0,0
4,2,0,1,0
5,2,0,1,1
6,1,0,1,1
7,0,1,0,0
8,0,0,1,0
9,2,1,1,0


In [9]:
tree = DecisionTreeClassifier(criterion='gini', random_state=0)
tree.fit(x, y)

print(export_text(tree, feature_names=list(maps)))

|--- student <= 0.50
|   |--- age <= 0.50
|   |   |--- class: no
|   |--- age >  0.50
|   |   |--- age <= 1.50
|   |   |   |--- class: yes
|   |   |--- age >  1.50
|   |   |   |--- credit_rating <= 0.50
|   |   |   |   |--- class: yes
|   |   |   |--- credit_rating >  0.50
|   |   |   |   |--- class: no
|--- student >  0.50
|   |--- credit_rating <= 0.50
|   |   |--- class: yes
|   |--- credit_rating >  0.50
|   |   |--- age <= 1.50
|   |   |   |--- class: yes
|   |   |--- age >  1.50
|   |   |   |--- class: no



In [12]:
instance = pd.DataFrame([{"age": 0, "income": 1, "student": 1, "credit_rating": 0}])
tree.predict(instance)

array(['yes'], dtype=object)

In [13]:
nb = CategoricalNB()
nb.fit(x, y)
nb.predict(instance)

array(['yes'], dtype='<U3')

In [14]:
proba = nb.predict_proba(instance)[0]

for label, p in zip(nb.classes_, proba):
    print(f"{label}: {p:.2f}")

no: 0.23
yes: 0.77
